In [ ]:
# =============================================================================
# EXPERIMENT SMALL - Quick Benchmarking Across All Datasets & Methods
# =============================================================================
# 
# Purpose: Run all methods on all datasets with reduced settings for fast testing
# - 1 CV fold only
# - 5000 row limit per dataset
# - Excludes slow methods (TabNet, MLP)
# - Produces AUC matrix for PD, R² matrix for LGD
#
# =============================================================================

# -----------------------------------------------------------------------------
# SETUP
# -----------------------------------------------------------------------------

import sys
from pathlib import Path
import pickle
import json
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

# Fixed settings for quick testing
ROW_LIMIT = 5000
CV_SPLITS = 1
MAX_EPOCHS = 15
BATCH_SIZE = 1024
TUNE = False
SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2

# Methods to exclude (too slow for quick testing)
EXCLUDE_METHODS = ['tabnet', 'mlp', 'TabNet', 'MLP']

print(f"\n{'='*80}")
print(f" EXPERIMENT SMALL - Quick Benchmarking")
print(f"{'='*80}")
print(f"  Row limit:       {ROW_LIMIT}")
print(f"  CV splits:       {CV_SPLITS}")
print(f"  Max epochs:      {MAX_EPOCHS}")
print(f"  HPO:             {TUNE}")
print(f"  Excluded:        {', '.join(EXCLUDE_METHODS)}")
print(f"{'='*80}\n")

# -----------------------------------------------------------------------------
# LOAD CONFIGURATIONS
# -----------------------------------------------------------------------------

from src.utils.config_reader import load_config

# Load unified config
config = load_config()

# Extract datasets
pd_datasets = list(config['datasets']['pd'].keys())
lgd_datasets = list(config['datasets']['lgd'].keys())

print(f"PD Datasets ({len(pd_datasets)}): {pd_datasets}")
print(f"LGD Datasets ({len(lgd_datasets)}): {lgd_datasets}")
print()

# Extract and filter methods
pd_methods_raw = list(config['methods']['pd'].keys())
lgd_methods_raw = list(config['methods']['lgd'].keys())

pd_methods = [m for m in pd_methods_raw if m not in EXCLUDE_METHODS]
lgd_methods = [m for m in lgd_methods_raw if m not in EXCLUDE_METHODS]

print(f"PD Methods ({len(pd_methods)}): {pd_methods}")
print(f"LGD Methods ({len(lgd_methods)}): {lgd_methods}")
print()

# -----------------------------------------------------------------------------
# RUN EXPERIMENTS
# -----------------------------------------------------------------------------

from src.methods.method_runner import run_talent_method

def run_single_experiment(task, dataset, method):
    """
    Run a single method-dataset combination and extract metric.
    
    Returns:
        float: AUC for PD task, R2 for LGD task
    """
    try:
        # Call run_talent_method with correct parameters
        results = run_talent_method(
            task=task,
            dataset=dataset,
            test_size=TEST_SIZE,
            val_size=VAL_SIZE,
            cv_splits=CV_SPLITS,
            seed=SEED,
            row_limit=ROW_LIMIT,
            method=method,
            max_epoch=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            tune=TUNE,
            n_trials=20,  # Not used since tune=False
            early_stopping=True,
            early_stopping_patience=10,
            verbose=False,
        )
        
        # Results structure: {fold_id: {metrics, y_true, y_prob, ...}}
        # With CV_SPLITS=1, we only have fold 1
        if 1 in results:
            fold_results = results[1]
            metrics = fold_results['metrics']
            
            if task == 'pd':
                # For PD: extract AUC
                metric = metrics.get('AUC', np.nan)
            else:  # lgd
                # For LGD: extract R2
                metric = metrics.get('R2', np.nan)
            
            return metric
        else:
            print(f"  WARNING: No fold 1 in results for {method} on {dataset}")
            return np.nan
            
    except Exception as e:
        print(f"  ERROR: {method} on {dataset} - {str(e)[:100]}")
        import traceback
        traceback.print_exc()
        return np.nan


def run_task_experiments(task, datasets, methods):
    """Run all method-dataset combinations for a task."""
    metric_name = "AUC" if task == 'pd' else "R2"
    
    print(f"\n{'='*80}")
    print(f" Running {task.upper()} Experiments")
    print(f"{'='*80}")
    print(f"  Datasets: {len(datasets)}")
    print(f"  Methods:  {len(methods)}")
    print(f"  Total:    {len(datasets) * len(methods)} experiments")
    print(f"  Metric:   {metric_name}")
    print(f"{'='*80}\n")
    
    # Initialize results matrix (datasets as rows, methods as columns)
    results_matrix = pd.DataFrame(index=datasets, columns=methods, dtype=float)
    
    # Total experiments
    total_experiments = len(datasets) * len(methods)
    
    # Progress bar
    with tqdm(total=total_experiments, desc=f"{task.upper()} Progress") as pbar:
        for dataset in datasets:
            for method in methods:
                pbar.set_description(f"{task.upper()}: {dataset[:20]:20s} + {method[:15]:15s}")
                metric = run_single_experiment(task, dataset, method)
                results_matrix.loc[dataset, method] = metric
                pbar.update(1)
    
    return results_matrix


# Run PD experiments
pd_results = run_task_experiments('pd', pd_datasets, pd_methods)

# Run LGD experiments  
lgd_results = run_task_experiments('lgd', lgd_datasets, lgd_methods)

print("\n✓ All experiments completed!")

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------

print(f"\n{'='*80}")
print(f" PD RESULTS (AUC)")
print(f"{'='*80}\n")
print(pd_results.to_string())

print(f"\n{'='*80}")
print(f" LGD RESULTS (R²)")
print(f"{'='*80}\n")
print(lgd_results.to_string())

# -----------------------------------------------------------------------------
# SAVE RESULTS
# -----------------------------------------------------------------------------

# Create output directory
output_dir = PROJECT_ROOT / 'results' / 'experiment_small'
output_dir.mkdir(parents=True, exist_ok=True)

# Generate timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save PD results (datasets as rows, methods as columns)
pd_csv_path = output_dir / f"pd_results_auc_{timestamp}.csv"
pd_results.to_csv(pd_csv_path)
print(f"\n✓ PD results saved to: {pd_csv_path}")

# Save LGD results (datasets as rows, methods as columns)
lgd_csv_path = output_dir / f"lgd_results_r2_{timestamp}.csv"
lgd_results.to_csv(lgd_csv_path)
print(f"✓ LGD results saved to: {lgd_csv_path}")

# Save configuration summary
config_summary = {
    'timestamp': timestamp,
    'settings': {
        'row_limit': ROW_LIMIT,
        'cv_splits': CV_SPLITS,
        'max_epochs': MAX_EPOCHS,
        'batch_size': BATCH_SIZE,
        'tune': TUNE,
        'seed': SEED,
        'test_size': TEST_SIZE,
        'val_size': VAL_SIZE,
        'excluded_methods': EXCLUDE_METHODS,
    },
    'datasets': {
        'pd': pd_datasets,
        'lgd': lgd_datasets,
    },
    'methods': {
        'pd': pd_methods,
        'lgd': lgd_methods,
    },
    'metrics': {
        'pd': 'AUC',
        'lgd': 'R²'
    },
    'n_experiments': len(pd_datasets) * len(pd_methods) + len(lgd_datasets) * len(lgd_methods),
}

config_path = output_dir / f"config_{timestamp}.json"
with open(config_path, 'w') as f:
    json.dump(config_summary, f, indent=2)
print(f"✓ Configuration saved to: {config_path}")

# -----------------------------------------------------------------------------
# SUMMARY STATISTICS
# -----------------------------------------------------------------------------

print(f"\n{'='*80}")
print(f" SUMMARY STATISTICS")
print(f"{'='*80}")

# PD Summary
if len(pd_results) > 0 and not pd_results.empty:
    print(f"\n[PD TASK - AUC]")
    print(f"  Best method overall:     {pd_results.mean(axis=0).idxmax()} (Avg: {pd_results.mean(axis=0).max():.4f})")
    print(f"  Worst method overall:    {pd_results.mean(axis=0).idxmin()} (Avg: {pd_results.mean(axis=0).min():.4f})")
    print(f"  Hardest dataset:         {pd_results.mean(axis=1).idxmin()} (Avg: {pd_results.mean(axis=1).min():.4f})")
    print(f"  Easiest dataset:         {pd_results.mean(axis=1).idxmax()} (Avg: {pd_results.mean(axis=1).max():.4f})")
    print(f"  Experiments completed:   {pd_results.notna().sum().sum()} / {pd_results.size}")
    print(f"  Failed experiments:      {pd_results.isna().sum().sum()}")

# LGD Summary
if len(lgd_results) > 0 and not lgd_results.empty:
    print(f"\n[LGD TASK - R²]")
    print(f"  Best method overall:     {lgd_results.mean(axis=0).idxmax()} (Avg: {lgd_results.mean(axis=0).max():.4f})")
    print(f"  Worst method overall:    {lgd_results.mean(axis=0).idxmin()} (Avg: {lgd_results.mean(axis=0).min():.4f})")
    print(f"  Hardest dataset:         {lgd_results.mean(axis=1).idxmin()} (Avg: {lgd_results.mean(axis=1).min():.4f})")
    print(f"  Easiest dataset:         {lgd_results.mean(axis=1).idxmax()} (Avg: {lgd_results.mean(axis=1).max():.4f})")
    print(f"  Experiments completed:   {lgd_results.notna().sum().sum()} / {lgd_results.size}")
    print(f"  Failed experiments:      {lgd_results.isna().sum().sum()}")

# -----------------------------------------------------------------------------
# VISUALIZATIONS - Simple Heatmaps Only
# -----------------------------------------------------------------------------

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# PD Heatmap (Datasets as rows, Methods as columns)
if len(pd_results) > 0 and not pd_results.empty:
    fig, ax = plt.subplots(figsize=(max(14, len(pd_methods) * 1.0), 
                                     max(8, len(pd_datasets) * 0.5)))
    
    sns.heatmap(
        pd_results.astype(float), 
        annot=True, 
        fmt='.3f', 
        cmap='RdYlGn',
        center=0.75, 
        vmin=0.5, 
        vmax=1.0,
        cbar_kws={'label': 'AUC'},
        linewidths=0.5,
        ax=ax
    )
    
    ax.set_title(
        f'PD Results - AUC Scores\n(Row limit: {ROW_LIMIT}, CV folds: {CV_SPLITS})',
        fontsize=14,
        fontweight='bold',
        pad=20
    )
    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    heatmap_path = output_dir / f"pd_heatmap_{timestamp}.png"
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    print(f"\n✓ PD heatmap saved to: {heatmap_path}")
    plt.show()

# LGD Heatmap (Datasets as rows, Methods as columns)
if len(lgd_results) > 0 and not lgd_results.empty:
    fig, ax = plt.subplots(figsize=(max(14, len(lgd_methods) * 1.0),
                                     max(8, len(lgd_datasets) * 0.5)))
    
    sns.heatmap(
        lgd_results.astype(float),
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        center=0.5,
        vmin=0.0,
        vmax=1.0,
        cbar_kws={'label': 'R²'},
        linewidths=0.5,
        ax=ax
    )
    
    ax.set_title(
        f'LGD Results - R² Scores\n(Row limit: {ROW_LIMIT}, CV folds: {CV_SPLITS})',
        fontsize=14,
        fontweight='bold',
        pad=20
    )
    ax.set_xlabel('Method', fontsize=12, fontweight='bold')
    ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
    
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    heatmap_path = output_dir / f"lgd_heatmap_{timestamp}.png"
    plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
    print(f"✓ LGD heatmap saved to: {heatmap_path}")
    plt.show()

print(f"\n{'='*80}")
print(f" EXPERIMENT COMPLETE")
print(f"{'='*80}")
print(f"  Timestamp: {datetime.now()}")
print(f"  Results saved in: {output_dir}")
print(f"{'='*80}")

c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: c:\Users\U0152019\OneDrive - KU Leuven\PhD Documents\Projects\1. TabPFN\TabPFNCredit

 EXPERIMENT SMALL - Quick Benchmarking
  Row limit:       5000
  CV splits:       1
  Max epochs:      15
  HPO:             False
  Excluded:        tabnet, mlp, TabNet, MLP

PD Datasets (15): ['0001.gmsc', '0002.taiwan_creditcard', '0003.vehicle_loan', '0004.lendingclub', '0005.case_study', '0006.myhom', '0007.hackerearth', '0008.cobranded', '0009.german', '0010.bank_status', '0011.thomas', '0012.loan_default', '0013.home_credit', '0014.hmeq', '0015.algorithmwatch']
LGD Datasets (7): ['0001.heloc', '0002.loss2', '0003.axa', '0004.base_model', '0005.base_modelisation', '0006.lgd_freddie', '0007.lgd_lendingclub']

PD Methods (12): ['catboost', 'knn', 'lightgbm', 'LogReg', 'NaiveBayes', 'RandomForest', 'svm', 'xgboost', 'NCM', 'dummy', 'tabpfn', 'tabpfn_v2']
LGD Methods (8): ['catboost', 'knn', 'lightgbm', 'LinearRegression', 'RandomForest', 'xgboost', 'svm', 'tabpfn_v2']


 Running PD Ex

PD: 0003.vehicle_loan    + tabpfn         :  19%|█▉        | 34/180 [09:19<11:06,  4.56s/it]  